# **Data Pipelines y ETLs**

**🎓 Objetivos :**

1. Comprender qué es un *data pipeline* y cómo se usa en entornos reales.
2. Identificar las etapas y funciones principales de un proceso ETL.
3. Diferenciar entre ETL y ELT, y cuándo usar cada uno.
4. Explorar herramientas y buenas prácticas para construir pipelines robustos y eficientes.


## **🔍 ¿Qué es un Data Pipeline y por qué es importante?**

Un *data pipeline* es un flujo automatizado que transporta datos desde su origen hasta su destino, pasando por una o varias etapas de transformación, validación y almacenamiento. Imagina una cadena de montaje 🏭 donde los datos crudos son recogidos, procesados y entregados listos para su análisis o uso en aplicaciones.

Los data pipelines son esenciales en proyectos de datos modernos porque permiten:
* Automatizar tareas repetitivas.
* Garantizar consistencia y calidad en los datos.
* Reducir errores humanos.
* Escalar el procesamiento de datos de forma eficiente ⚙️

---

## **🧪 ¿Qué es una ETL?**

ETL es un acrónimo que representa tres pasos fundamentales en el procesamiento de datos:

1. **Extract (Extracción):**  
   Se refiere a obtener datos desde una o más fuentes, como bases de datos, archivos CSV, APIs o servicios externos. Por ejemplo, podríamos extraer ventas desde una hoja de cálculo en Google Sheets o datos de usuarios desde una API REST.

2. **Transform (Transformación):**  
   Aquí los datos se limpian, normalizan y ajustan según las necesidades del negocio. Esto puede incluir cambios de formato, eliminación de valores nulos, creación de nuevas columnas derivadas o cruces entre tablas.  
   Es como preparar los ingredientes antes de cocinar 🍳: quitar lo que no sirve y dejar todo listo para el análisis.

3. **Load (Carga):**  
   Finalmente, los datos transformados se cargan en un destino, como un almacén de datos (*data warehouse*), una base de datos analítica o incluso una hoja de cálculo para visualización. Esta es la parte donde los datos ya están listos para usarse 🔎📊

## **🛠️ Actividad práctica: Construyendo un ETL con SQLite usando datos de Sakila**

En esta actividad vamos a simular un pequeño **proceso ETL** utilizando la base de datos *Sakila*. A lo largo del ejercicio construiremos una nueva base de datos, limpia y lista para análisis, aplicando los pasos clásicos de **Extract**, **Transform** y **Load**.


---

### **📌 Pasos del ejercicio:**

1. **Extracción:**  
   Conectarse a la base de datos original de Sakila y realizar consultas SQL para extraer la información relevante.

2. **Transformación:**  
   Limpiar, filtrar o enriquecer los datos. Por ejemplo:
   * Crear columnas nuevas como año de renta.
   * Filtrar pagos por monto.
   * Agrupar datos para identificar patrones.

3. **Carga:**  
   Crear una nueva base de datos (`sakila_etl_output.db`) y almacenar allí las tablas generadas.

---

### **📊 Tablas que vamos a construir:**

*1. `rentas_filtradas`*  
Contiene las rentas con pagos mayores a 5 USD, junto con el nombre del cliente y la fecha de renta.

*2. `top_clientes`*  
Clientes que han pagado más de $100 en total. Útil para entender quiénes son los más valiosos para el negocio.

*3. `peliculas_mas_rentadas`*  
Películas que se han rentado más de 30 veces. Una forma de identificar el catálogo más exitoso.

*4. `actividades_mensuales`*  
Número total de rentas y pagos por mes. Ideal para hacer visualizaciones de actividad mensual o identificar estacionalidades.

---




### **Extracción**

Vamos a extraer información clave desde la base de datos original de Sakila

---

In [ ]:
import sqlite3
import pandas as pd

In [ ]:
# Paso 1: Clonar el repositorio
!git clone https://github.com/bradleygrant/sakila-sqlite3.git

In [ ]:
# Crear conexión
conn = sqlite3.connect("/content/sakila-sqlite3/sakila_master.db")

In [ ]:
# Ver tablas disponibles
query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql(query, conn)
print("Tablas en la base de datos:")
print(tables)


In [ ]:
conn.close()

In [ ]:
def run_query(query):
  query_parsed= query.replace('\xa0', ' ').replace('\n',' ').strip()
  conn = sqlite3.connect("/content/sakila-sqlite3/sakila_master.db")
  df = pd.read_sql(query_parsed, conn)
  conn.close()
  return df

#### Descripción de las tablas principales de la base de datos Sakila y sus relaciones

- **actor**: Contiene información de los actores.  
  - Clave primaria: `actor_id`

- **country**: Lista de países.  
  - Clave primaria: `country_id`

- **city**: Lista de ciudades, cada una asociada a un país.  
  - Clave primaria: `city_id`  
  - Clave foránea: `country_id` → `country.country_id`

- **address**: Direcciones físicas, asociadas a una ciudad.  
  - Clave primaria: `address_id`  
  - Clave foránea: `city_id` → `city.city_id`

- **language**: Idiomas disponibles para las películas.  
  - Clave primaria: `language_id`

- **category**: Categorías de películas (géneros).  
  - Clave primaria: `category_id`

- **customer**: Información de clientes, cada uno asociado a una dirección y tienda.  
  - Clave primaria: `customer_id`  
  - Claves foráneas:  
    - `address_id` → `address.address_id`  
    - `store_id` → `store.store_id`

- **film**: Información de películas, incluyendo idioma y detalles generales.  
  - Clave primaria: `film_id`  
  - Claves foráneas:  
    - `language_id` → `language.language_id`  
    - `original_language_id` → `language.language_id` (puede ser NULL)

- **film_actor**: Relaciona películas con actores (muchos a muchos).  
  - Clave primaria compuesta: `film_id`, `actor_id`  
  - Claves foráneas:  
    - `film_id` → `film.film_id`  
    - `actor_id` → `actor.actor_id`

- **film_category**: Relaciona películas con categorías (muchos a muchos).  
  - Clave primaria compuesta: `film_id`, `category_id`  
  - Claves foráneas:  
    - `film_id` → `film.film_id`  
    - `category_id` → `category.category_id`

- **film_text**: Texto de búsqueda para películas (sin claves foráneas directas).

- **inventory**: Inventario de copias de películas en cada tienda.  
  - Clave primaria: `inventory_id`  
  - Claves foráneas:  
    - `film_id` → `film.film_id`  
    - `store_id` → `store.store_id`

- **staff**: Empleados de la tienda, cada uno asociado a una dirección y tienda.  
  - Clave primaria: `staff_id`  
  - Claves foráneas:  
    - `address_id` → `address.address_id`  
    - `store_id` → `store.store_id`

- **store**: Tiendas físicas, cada una asociada a una dirección y un gerente (staff).  
  - Clave primaria: `store_id`  
  - Claves foráneas:  
    - `address_id` → `address.address_id`  
    - `manager_staff_id` → `staff.staff_id`

- **payment**: Pagos realizados por clientes.  
  - Clave primaria: `payment_id`  
  - Claves foráneas:  
    - `customer_id` → `customer.customer_id`  
    - `staff_id` → `staff.staff_id`  
    - `rental_id` → `rental.rental_id`

- **rental**: Rentas de películas por parte de clientes.  
  - Clave primaria: `rental_id`  
  - Claves foráneas:  
    - `inventory_id` → `inventory.inventory_id`  
    - `customer_id` → `customer.customer_id`  
    - `staff_id` → `staff.staff_id`

---

**Resumen de relaciones:**  
- Las tablas principales (`film`, `customer`, `store`, `staff`) se conectan a través de tablas de relación como `film_actor`, `film_category`, `inventory` y `rental`.
- Las claves foráneas aseguran la integridad referencial y permiten consultas complejas combinando información de varias

#### 🗂️ Extracción 1: `rentas_filtradas`

Rentas con pagos mayores a 5 USD, incluyendo cliente y fechas.

In [ ]:
query_rentas = """
SELECT r.rental_id, r.rental_date, c.first_name || ' ' || c.last_name AS customer_name,
       p.amount, p.payment_date
FROM rental r
JOIN customer c ON r.customer_id = c.customer_id
JOIN payment p ON r.rental_id = p.rental_id
"""
df_rentas = run_query(query_rentas) 

#### 🧾 Extracción 2: `top_clientes`

Clientes que han pagado más de $100 en total.

In [ ]:
query_top_clientes = """
SELECT c.customer_id, c.first_name || ' ' || c.last_name AS cliente,
       SUM(p.amount) AS total_pagado
FROM payment p
JOIN customer c ON p.customer_id = c.customer_id
GROUP BY c.customer_id
HAVING total_pagado > 100
"""
df_top_clientes = run_query(query_top_clientes) 


### 🎬 Extracción 3: `peliculas_mas_rentadas`

Películas que han sido rentadas más de 30 veces.

In [ ]:
query_peliculas = """
SELECT f.film_id, f.title, COUNT(*) AS total_rentas
FROM rental r
JOIN inventory i ON r.inventory_id = i.inventory_id
JOIN film f ON i.film_id = f.film_id
GROUP BY f.film_id
HAVING total_rentas > 30
"""
df_peliculas = run_query(query_peliculas) 

### 📊 Extracción 4: `actividades_mensuales`

Rentas y pagos agrupados por mes.

In [ ]:
query_mensual = """
SELECT 
    strftime('%Y-%m', r.rental_date) AS mes,
    COUNT(DISTINCT r.rental_id) AS total_rentas,
    COUNT(DISTINCT p.payment_id) AS total_pagos
FROM rental r
JOIN payment p ON r.rental_id = p.rental_id
GROUP BY mes
ORDER BY mes
"""
df_mensual = run_query(query_mensual) 


### Transformación 

En esta etapa vamos a limpiar, enriquecer o filtrar los datos extraídos de la base `sakila_master.db` para prepararlos antes de su carga. Estas transformaciones ayudan a tener datos más útiles y coherentes para análisis o visualización.

---

### 🗂️ Transformación 1: `rentas_filtradas`

Filtraremos solo las rentas con pagos mayores a 5 USD y extraeremos el año de la renta para futuros análisis temporales.

In [ ]:
# Convertir fechas y extraer año
df_rentas['rental_date'] = pd.to_datetime(df_rentas['rental_date'])
df_rentas['rental_year'] = df_rentas['rental_date'].dt.year

# Filtrar por pagos mayores a 5 USD
df_rentas_filtradas = df_rentas[df_rentas['amount'] > 5].copy()

### 🧾 Transformación 2: `top_clientes`

Redondeamos el total pagado a 2 decimales y ordenamos los clientes por mayor contribución.

In [ ]:
df_top_clientes['total_pagado'] = df_top_clientes['total_pagado'].round(2)
df_top_clientes = df_top_clientes.sort_values(by='total_pagado', ascending=False).reset_index(drop=True)

### 🎬 Transformación 3: `peliculas_mas_rentadas`

Agregamos una clasificación rápida del nivel de popularidad basado en número de rentas.

In [ ]:
def clasificar_popularidad(x):
    if x > 100:
        return "🔥 Éxito total"
    elif x > 50:
        return "📈 Muy popular"
    else:
        return "📊 Popular"

df_peliculas['popularidad'] = df_peliculas['total_rentas'].apply(clasificar_popularidad)

### 📊 Transformación 4: `actividades_mensuales`

Convertimos las columnas de conteo a tipo entero y renombramos las columnas para mayor claridad.

In [ ]:
df_mensual['total_rentas'] = df_mensual['total_rentas'].astype(int)
df_mensual['total_pagos'] = df_mensual['total_pagos'].astype(int)
df_mensual = df_mensual.rename(columns={'mes': 'mes_anio'})

Estas transformaciones preparan los datos para que estén listos para cargar en la nueva base, y también mejoran su utilidad para visualizaciones, dashboards o reportes. ¿Avanzamos con la sección de carga ahora? 💾📤


### Carga de datos 

En esta etapa final vamos a guardar las tablas transformadas en una nueva base de datos SQLite llamada `sakila_etl_output.db`. Esto nos permitirá consultarlas posteriormente con SQL o integrarlas en dashboards.

---

In [ ]:
# Crear conexión a la nueva base
conn_etl = sqlite3.connect("sakila_etl_output.db")

In [ ]:
def guardar_tabla(df, nombre_tabla, conn, modo='replace'):
    """
    Guarda un DataFrame en la base SQLite especificada.

    Parámetros:
    - df: DataFrame a guardar.
    - nombre_tabla: nombre de la tabla destino.
    - conn: conexión sqlite3.
    - modo: 'replace' para sobrescribir, 'append' para agregar filas.
    """
    df.to_sql(nombre_tabla, conn, index=False, if_exists=modo)
    print(f"✅ Tabla '{nombre_tabla}' guardada ({modo})")

In [ ]:
guardar_tabla(df_rentas_filtradas, 'rentas_filtradas', conn_etl)

# Top clientes
guardar_tabla(df_top_clientes, 'top_clientes', conn_etl)

# Películas más rentadas
guardar_tabla(df_peliculas, 'peliculas_mas_rentadas', conn_etl)

# Actividades mensuales
guardar_tabla(df_mensual, 'actividades_mensuales', conn_etl)

### 📋 Verificación final

---

In [ ]:
# Consultar las tablas disponibles en la nueva base
tablas_cargadas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn_etl)
print("Tablas en la nueva base:")
print(tablas_cargadas)

## **📡 Haciendo los datos disponibles**

Una vez que los datos están limpios y almacenados en una base organizada, su verdadero valor emerge cuando **se ponen al servicio de otras áreas** del negocio. Ya sea para el equipo de marketing, ventas, finanzas o producto, los datos deben ser fáciles de consultar, entender y analizar.

Aquí es donde entran los **dashboards** 🎛️.

---

### 📊 ¿Qué es un dashboard?

Un *dashboard* es una interfaz visual interactiva que permite **consultar información clave en tiempo real o con datos actualizados**, generalmente usando gráficos, tablas y filtros. Son una herramienta poderosa para:

* Tomar decisiones informadas rápidamente 🚀  
* Comunicar resultados de forma efectiva 🧠  
* Detectar anomalías o patrones visualmente 👀  
* Automatizar reportes recurrentes 🗓️

---



### 🧪 Actividad práctica: ¡Lanza tu primer dashboard con Dash!

Usaremos la librería **Dash** para crear un dashboard interactivo que lea directamente los datos de nuestra base `sakila_etl_output.db`.

🔧 Verás lo sencillo que es:

* Elegir qué tabla mostrar.
* Aplicar filtros como selección de año, cliente o película.
* Visualizar métricas clave como total de rentas, ingresos o actividad mensual.

---

✅ ¡Así terminamos el ciclo completo de un proyecto de datos real!
De la extracción → transformación → carga → visualización.



**Creemos el dashboard más simple que se pueda hacer en Dash**

In [ ]:
#!pip install dash


In [ ]:
from dash import Dash, html

# Crear la app
app = Dash(__name__)

# Definir el layout
app.layout = html.Div([
    html.H1("📊 Dashboard de ejemplo", style={'textAlign': 'center'}),
    html.H3("Este es un subtítulo sencillo para comenzar", style={'textAlign': 'center', 'color': 'gray'})
])

# Ejecutar el servidor (modo notebook si estás en Jupyter)
app.run(mode='inline')


**Ahora usemos los datos que hemos guardado para generar visualizaciones**

In [ ]:
import sqlite3
import pandas as pd
from dash import Dash, html, dcc, Input, Output, dash_table
import plotly.express as px

In [ ]:
# Conectar a la base sakila_etl_output.db
conn = sqlite3.connect("sakila_etl_output.db")

# Leer datos de actividades mensuales
df = pd.read_sql("SELECT * FROM actividades_mensuales", conn)

# Crear app Dash
app = Dash(__name__)

# Layout del dashboard
app.layout = html.Div([
    html.H1("📊 Dashboard de Actividad Mensual", style={'textAlign': 'center'}),
    html.H3("Basado en datos de rentas y pagos por mes (ETL Sakila)", style={'textAlign': 'center', 'color': 'gray'}),
    
    html.Label("Selecciona el año:"),
    dcc.Dropdown(
        options=[{"label": y, "value": y} for y in sorted(df["mes_anio"].str[:4].unique())],
        value=df["mes_anio"].str[:4].min(),
        id="filtro_anio"
    ),
    
    html.Br(),

    dash_table.DataTable(
        id='tabla_mensual',
        columns=[{"name": col, "id": col} for col in df.columns],
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'center'},
        page_size=12
    ),

    html.Br(),

    dcc.Graph(id="grafico_mensual")
])

# Callback para actualizar tabla y gráfico
@app.callback(
    Output("tabla_mensual", "data"),
    Output("grafico_mensual", "figure"),
    Input("filtro_anio", "value")
)
def actualizar_dashboard(anio_seleccionado):
    df_filtrado = df[df["mes_anio"].str.startswith(anio_seleccionado)]

    fig = px.bar(df_filtrado,
                 x="mes_anio",
                 y=["total_rentas", "total_pagos"],
                 barmode="group",
                 title=f"Actividad mensual en {anio_seleccionado}",
                 labels={"value": "Cantidad", "mes_anio": "Mes"},
                 height=400)
    
    return df_filtrado.to_dict("records"), fig

# Ejecutar en el notebook
app.run(mode="inline")

> Si quieres saber un poco más sobre qué puedes hacer con Dash, visita [Dash Gallery](https://dash.gallery/dash-world-cell-towers/)

## Antes de cerrar, me gustaría saber tu opinión: 😊✨

----
1. ¿Qué ventajas trae separar la lógica en pasos *Extract*, *Transform* y *Load*?
2. ¿Qué retos podrías encontrar si tuvieras que automatizar este proceso con millones de registros?
3. ¿Qué herramientas usarías si la fuente fuera una API o archivos CSV diarios?
4. ¿Qué beneficios ofrece visualizar los datos en un dashboard en lugar de solo revisar tablas?


## 🚀 Para seguir aprendiendo :

---

- 📚 Vuelve a revisar este notebook y trata resolver por tu cuenta el proyecto nuevamente
- 💬 Recuerda que en Discord puedes dejar todos tus comentarios y dudas sobre el contenido del sprint en [Sprint 12](https://discord.com/channels/1081207584104656986/1270074296395497513).
    - 📝 Si tienes preguntas sobre tu proyecto, usa el canal `#project` para recibir ayuda y compartir ideas.
    - 🤝 Aprovecha el espacio de `CoLearning` para aclarar tus dudas junto con otros estudiantes e instructores: [Co-Learning](https://discord.com/channels/1081207584104656986/1197953851391746119).
- 📅 ¿Necesitas ayuda personalizada? Puedes agendar una sesión `1:1` conmigo aquí: [1:1 Roman Castillo](https://scheduler.zoom.us/roman-castillo/1-1-roman-castillo).

- Por último hazme paro y responde la encuesta al final de la sesión, me sirve para poder ayudarte mejor 

¡Sigue practicando y no dudes en pedir apoyo cuando lo necesites! 💪✨